**News content binary coding tool using ChatGPT**

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 이진 코딩 분석기 (Binary Coding Analyzer)
# ═══════════════════════════════════════════════════════════════════════

import pandas as pd
from openai import AsyncOpenAI
import asyncio
import json
import os
import logging
import time
from typing import Tuple, Dict
import re
from tqdm import tqdm

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 1. 분석 기준 정의(맥락 제공, 기준 설명, 판단 기준, 예시 포함, LLM을 교육하는 용도로 생각)
# ═════════════════════════════════════════════════════════════════════════════════
# 아래 [대괄호] 부분을 본인의 연구 주제에 맞게 채워 넣으세요.
ANALYSIS_CRITERIA = """
**[코딩 대상 항목] 판단 기준:**

이 분석은 기사가 "[코딩 대상 항목]"에 해당하는지 판단합니다.

**배경:**
- [분석 맥락과 배경을 설명하세요]
- [코딩 대상 항목의 정의를 설명하세요]

**"예(1)"로 판단하는 경우 (다음 중 하나라도 해당):**

1. **[판단 기준 1]**
   - [세부 기준 또는 예시]
   - [세부 기준 또는 예시]

2. **[판단 기준 2]**
   - [세부 기준 또는 예시]
   - [세부 기준 또는 예시]

3. **[판단 기준 3]**
   - [세부 기준 또는 예시]
   - [세부 기준 또는 예시]

**중요:** 기사 본문에서 직접 언급하거나, 정보원(전문가, 정치인, 학자 등)의 발언을 인용하여 간접적으로 언급해도 "예(1)"로 판단

**"아니오(0)"로 판단하는 경우:**
- 단순 사실 전달만 하는 경우
- [코딩 대상 항목]과 무관한 내용
"""


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 2. 프롬프트 템플릿
# ═══════════════════════════════════════════════════════════════════════
PROMPT_TEMPLATE = """다음 뉴스 기사가 "[코딩 대상 항목]"에 해당하는지 판단해주세요.

[뉴스 기사]:
{text}

{criteria}

**분석 방법:**
1. 기사에서 [코딩 대상 항목]과 관련된 표현을 찾으세요
2. 기사 본문의 직접적 언급 또는 정보원 발언의 간접적 언급 모두 확인하세요
3. 위 기준의 판단 요소 중 해당하는 내용이 있는지 판단하세요
4. 위 기준에 따라 "예(1)" 또는 "아니오(0)"를 선택하세요
5. 판단 근거를 구체적으로 제시하세요
6. 판단 신뢰도를 1-100 사이의 정수로 평가하세요

**반드시 다음 JSON 형식으로만 답변하세요:**
{{
  "coding_result": 1 또는 0,
  "reason": "판단 근거 (기사 내 구체적 표현이나 정보원 발언 인용)",
  "confidence": 85
}}

답변 주의사항:
- coding_result 필드는 1(예) 또는 0(아니오) 중 정확히 하나만
- reason 필드는 기사에서 발견한 구체적 근거 제시
- confidence는 1-100 사이의 정수
- JSON 외 다른 내용 금지

JSON 응답:"""

# ═══════════════════════════════════════════════════════════════════════
# 3. JSON 파서
# ═══════════════════════════════════════════════════════════════════════
class EnhancedJSONParser:
    """강화된 JSON 파싱 클래스"""
    
    @staticmethod
    def extract_json_robust(response_text: str) -> Dict:
        """강화된 JSON 추출 로직"""
        if not response_text or not response_text.strip():
            return EnhancedJSONParser._get_default_result("빈 응답")
        
        response_text = response_text.strip()
        
        # 1단계: 직접 JSON 파싱 시도
        try:
            if response_text.startswith('{') and response_text.endswith('}'):
                return json.loads(response_text)
        except json.JSONDecodeError:
            pass
        
        # 2단계: 마크다운 JSON 블록 추출
        json_patterns = [
            r'```json\s*\n?(.*?)\n?```',
            r'```\s*\n?(.*?)\n?```',
            r'`{3,}\s*json\s*\n?(.*?)\n?`{3,}',
        ]
        
        for pattern in json_patterns:
            match = re.search(pattern, response_text, re.DOTALL | re.IGNORECASE)
            if match:
                json_str = match.group(1).strip()
                try:
                    return json.loads(json_str)
                except json.JSONDecodeError:
                    continue
        
        # 3단계: 중괄호 블록 추출
        brace_patterns = [
            r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}',
            r'\{(?:[^{}]|{[^{}]*})*\}',
        ]
        
        for pattern in brace_patterns:
            matches = re.findall(pattern, response_text)
            for match in matches:
                try:
                    result = json.loads(match)
                    if EnhancedJSONParser._validate_result(result):
                        return result
                except json.JSONDecodeError:
                    continue
        
        # 4단계: 키-값 개별 추출
        extracted_data = {}
        
        # coding_result 필드 추출
        warning_patterns = [
            r'"coding_result"\s*:\s*([01])',
            r'coding_result["\']?\s*[:=]\s*([01])',
        ]
        
        for pattern in warning_patterns:
            match = re.search(pattern, response_text, re.IGNORECASE)
            if match:
                extracted_data['coding_result'] = int(match.group(1))
                break
        
        # reason 필드 추출
        reason_patterns = [
            r'"reason"\s*:\s*"([^"]*(?:\\.[^"]*)*)"',
            r'reason["\']?\s*[:=]\s*["\']([^"\']+)["\']',
        ]
        
        for pattern in reason_patterns:
            match = re.search(pattern, response_text, re.IGNORECASE | re.DOTALL)
            if match:
                extracted_data['reason'] = match.group(1).strip()
                break
        
        # confidence 필드 추출
        confidence_patterns = [
            r'"confidence"\s*:\s*([0-9]+)',
            r'confidence["\']?\s*[:=]\s*([0-9]+)',
        ]
        
        for pattern in confidence_patterns:
            match = re.search(pattern, response_text, re.IGNORECASE)
            if match:
                try:
                    extracted_data['confidence'] = int(match.group(1))
                except ValueError:
                    pass
                break
        
        if extracted_data and 'coding_result' in extracted_data:
            result = {
                'coding_result': extracted_data['coding_result'],
                'reason': extracted_data.get('reason', '근거 추출 실패'),
                'confidence': extracted_data.get('confidence', 0)
            }
            return result
        
        # 5단계: 기본값 반환
        return EnhancedJSONParser._get_default_result("파싱 실패")
    
    @staticmethod
    def _validate_result(result: Dict) -> bool:
        """결과 유효성 검증"""
        if 'coding_result' not in result:
            return False
        if result['coding_result'] not in [0, 1]:
            return False
        return True
    
    @staticmethod
    def _get_default_result(reason: str) -> Dict:
        """기본 결과 반환"""
        return {
            'coding_result': -1,
            'reason': f'JSON 파싱 실패: {reason}',
            'confidence': 0
        }

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 4. 이진 코딩 분석기
# ═══════════════════════════════════════════════════════════════════════
class BinaryCodingAnalyzer:
    """이진 코딩 분석기"""
    
    def __init__(self, api_key, model, output_dir='.', 
                 concurrent_requests=3, rate_limit_delay=1.0, batch_size=5):
        self.client = AsyncOpenAI(api_key=api_key)
        self.model = model
        self.output_dir = output_dir
        self.concurrent_requests = concurrent_requests
        self.rate_limit_delay = rate_limit_delay
        self.batch_size = batch_size
        self.total_cost = 0.0
        self.semaphore = asyncio.Semaphore(concurrent_requests)
        self.json_parser = EnhancedJSONParser()
        
        # 통계
        self.parsing_successes = 0
        self.parsing_failures = 0
        
        # 로깅 설정 (WARNING 레벨로 최소화)
        logging.basicConfig(level=logging.WARNING, format='%(message)s')
        self.logger = logging.getLogger(__name__)
        
        os.makedirs(self.output_dir, exist_ok=True)
        
    def combine_title_text(self, title: str, text: str) -> str:
        """제목과 본문을 결합"""
        if pd.isna(title):
            title = ""
        if pd.isna(text):
            text = ""
        
        title = str(title).strip()
        text = str(text).strip()
        
        combined = f"{title}\n\n{text}" if title and text else (title or text)
        return combined[:4000]  # 토큰 제한을 위한 길이 제한

    async def analyze_single_article(self, title: str, text: str) -> Tuple[int, str, int, float]:
        """단일 기사 분석 - 각 호출은 완전히 독립적"""
        
        async with self.semaphore:
            try:
                # 제목과 본문 결합
                combined_text = self.combine_title_text(title, text)
                
                if not combined_text or combined_text == "":
                    return -1, "빈 텍스트", 0, 0.0
                
                # 프롬프트 생성 (매번 독립적으로 생성)
                prompt = PROMPT_TEMPLATE.format(
                    text=combined_text,
                    criteria=ANALYSIS_CRITERIA
                )
                
                # API 호출 (각 호출은 독립적인 세션)
                response = await self.client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {"role": "user", "content": prompt}
                    ],
                    max_tokens=500,
                    temperature=0,
                    response_format={"type": "json_object"}
                )
                
                response_text = response.choices[0].message.content.strip()
                result = self.json_parser.extract_json_robust(response_text)
                
                # 파싱 성공/실패 통계
                if result.get('coding_result') == -1:
                    self.parsing_failures += 1
                else:
                    self.parsing_successes += 1
                
                # 결과 추출
                coding_result = result.get("coding_result", -1)
                reason = result.get("reason", "근거 없음").replace('"reason":', '').replace('"', '').strip()
                confidence = int(result.get("confidence", 0))
                
                # 신뢰도 범위 검증
                confidence = max(0, min(100, confidence))
                
                # 비용 계산
                estimated_cost = 0.002
                self.total_cost += estimated_cost
                
                return coding_result, reason, confidence, estimated_cost
                
            except Exception as e:
                self.parsing_failures += 1
                return -1, f"오류: {str(e)[:50]}", 0, 0.0
            
            finally:
                await asyncio.sleep(self.rate_limit_delay)

    async def process_batch(self, batch_df) -> Tuple[list, list, list, list]:
        """배치 처리"""
        tasks = []
        
        for _, row in batch_df.iterrows():
            task = self.analyze_single_article(
                row['title'], 
                row['text']
            )
            tasks.append(task)
        
        results = await asyncio.gather(*tasks, return_exceptions=True)
        
        warnings = []
        reasons = []
        confidences = []
        costs = []
        
        for result in results:
            if isinstance(result, Exception):
                warnings.append(-1)
                reasons.append(f"배치 오류: {str(result)[:50]}")
                confidences.append(0)
                costs.append(0.0)
                self.parsing_failures += 1
            else:
                warnings.append(result[0])
                reasons.append(result[1])
                confidences.append(result[2])
                costs.append(result[3])
        
        return warnings, reasons, confidences, costs

    async def analyze_dataframe(self, df, output_filename="binary_coding_results.csv"):
        """전체 데이터프레임 분석"""
        
        output_path = os.path.join(self.output_dir, output_filename)
        
        all_warnings = []
        all_reasons = []
        all_confidences = []
        all_costs = []
        
        start_time = time.time()
        
        # tqdm 진행바 생성
        pbar = tqdm(total=len(df), desc="이진 코딩 분석 중", unit="건")
        
        # 배치별 처리
        for i in range(0, len(df), self.batch_size):
            batch_end = min(i + self.batch_size, len(df))
            batch_df = df.iloc[i:batch_end]
            
            try:
                warnings, reasons, confidences, costs = await self.process_batch(batch_df)
                
                all_warnings.extend(warnings)
                all_reasons.extend(reasons)
                all_confidences.extend(confidences)
                all_costs.extend(costs)
                
                # 진행바 업데이트
                pbar.update(len(batch_df))
                
                # 중간 저장 (5배치마다)
                batch_num = i // self.batch_size + 1
                if batch_num % 5 == 0:
                    temp_df = df.copy()
                    temp_df['코딩결과'] = all_warnings + [-1] * (len(df) - len(all_warnings))
                    temp_df['근거'] = all_reasons + [''] * (len(df) - len(all_reasons))
                    temp_df['신뢰도'] = all_confidences + [0] * (len(df) - len(all_confidences))
                    temp_df['비용'] = all_costs + [0.0] * (len(df) - len(all_costs))
                    temp_df.to_csv(output_path, encoding='utf-8-sig', index=False)
                
            except Exception as e:
                batch_size = len(batch_df)
                all_warnings.extend([-1] * batch_size)
                all_reasons.extend([f"배치 오류: {str(e)[:30]}"] * batch_size)
                all_confidences.extend([0] * batch_size)
                all_costs.extend([0.0] * batch_size)
                self.parsing_failures += batch_size
                pbar.update(len(batch_df))
        
        pbar.close()
        
        # 최종 결과 생성
        result_df = df.copy()
        result_df['코딩결과'] = all_warnings
        result_df['근거'] = all_reasons
        result_df['신뢰도'] = all_confidences
        result_df['비용'] = all_costs
        
        # 최종 저장
        result_df.to_csv(output_path, encoding='utf-8-sig', index=False)
        
        # 완료 메시지
        elapsed_time = time.time() - start_time
        self._print_completion_message(result_df, elapsed_time, output_path)
        
        return result_df

    def _print_completion_message(self, result_df, elapsed_time, output_path):
        """완료 메시지 출력"""
        total_count = len(result_df)
        
        # 유효한 분석 결과만 집계
        valid_results = result_df[result_df['코딩결과'].isin([0, 1])]
        yes_count = (valid_results['코딩결과'] == 1).sum()
        no_count = (valid_results['코딩결과'] == 0).sum()
        fail_count = (result_df['코딩결과'] == -1).sum()
        
        print(f"\n{'='*60}")
        print(f"✅ 이진 코딩 분석 완료!")
        print(f"{'='*60}")
        print(f"📊 총 {total_count:,}건 뉴스 분석 완료 (소요시간: {elapsed_time/60:.1f}분)")
        print(f"💰 총 비용: ${self.total_cost:.4f}")
        print(f"📁 결과 파일: {output_path}")
        print(f"\n📈 분석 결과:")
        if len(valid_results) > 0:
            yes_pct = (yes_count / len(valid_results)) * 100
            no_pct = (no_count / len(valid_results)) * 100
            print(f"   • 예(1) - 해당함: {yes_count}건 ({yes_pct:.1f}%)")
            print(f"   • 아니오(0) - 해당하지 않음: {no_count}건 ({no_pct:.1f}%)")
        if fail_count > 0:
            print(f"   • 분석 실패: {fail_count}건")
        print(f"{'='*60}\n")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 5. 설정 및 실행
# ═══════════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────────
# 여기서 설정을 변경하세요
# ──────────────────────────────────────────────────────────────────────
MODEL = "gpt-4.1-mini"  # "gpt-4o-mini" 또는 "gpt-4o"
API_KEY = "Your-OpenAI-API-Key-Here"
INPUT_CSV_PATH = "Your-Input-CSV-Path-Here"
OUTPUT_CSV_PATH = "Your-Output-CSV-Path-Here"
# ──────────────────────────────────────────────────────────────────────

async def main():
    """메인 함수"""
    try:
        # CSV 파일 로드
        df = pd.read_csv(INPUT_CSV_PATH, encoding='utf-8-sig')
        
        # 필수 컬럼 확인
        if 'title' not in df.columns or 'text' not in df.columns:
            print("❌ title, text 컬럼 필요")
            return
        
        # 출력 경로 설정
        output_dir = os.path.dirname(OUTPUT_CSV_PATH) or '.'
        output_file = os.path.basename(OUTPUT_CSV_PATH)
        
        # 분석기 생성 및 실행
        analyzer = BinaryCodingAnalyzer(
            api_key=API_KEY,
            model=MODEL,
            output_dir=output_dir
        )
        
        await analyzer.analyze_dataframe(df, output_file)
        
    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없음: {INPUT_CSV_PATH}")
    except Exception as e:
        print(f"❌ 오류: {str(e)}")

def run_analysis():
    """분석 실행"""
    try:
        import nest_asyncio
        nest_asyncio.apply()
        asyncio.run(main())
    except ImportError:
        asyncio.run(main())

if __name__ == "__main__":
    run_analysis()
